## 0. Load packages and dataset

### 0.0. Loading packages and setting Jupyter options

In [113]:
import numpy as np
import pooch
import scanpy as sc
import pandas as pd
import os
from plotnine import * 
import matplotlib.pyplot as plt


In [114]:
from IPython.core.interactiveshell import InteractiveShell
InteractiveShell.ast_node_interactivity = "all" # display all outputs

# Set display to show up to 100 rows and full string length
pd.set_option('display.max_rows', 100)
pd.set_option('display.max_colwidth', None)


In [115]:
#pd.options.mode.string_storage = "python"
#pd.options.mode.copy_on_write = False

### 0.1. Data loading

In [116]:
file_path = pooch.retrieve(
    url="https://datasets.cellxgene.cziscience.com/1dcf15ee-c103-4aaa-8b8c-0fc697fcccc8.h5ad",
    known_hash=None,  # Pooch will print the hash for you after the first download
    fname="cellxgene_intestine_oliver.h5ad",
    path="../data",
    progressbar=True
)

In [117]:
# read data as adata
adata = sc.read_h5ad("../data/cellxgene_intestine_oliver.h5ad")

In [118]:
# first inspection of adata structure
print(adata) # 1596200 cells × 18370 genes
print(adata.layers.keys()) # no keys
print(adata.X[:100, :100].toarray()) # X slot contains normalised data
print(adata.raw.X[:100, :100].toarray()) # raw.X slot contains raw data


AnnData object with n_obs × n_vars = 1596200 × 18370
    obs: 'sampleID', 'level_1_annot', 'level_2_annot', 'level_3_annot', 'n_counts', 'cell_type_ontology_term_id', 'sourceID', 'study', 'donorID_unified', 'donor_category', 'donor_disease', 'organ_unified', 'age_unified', 'sample_type', 'sample_category', 'sample_retrieval', 'tissue_fraction', 'cell_fraction_unified', 'cell_sorting', 'organ_groups', 'control_vs_disease', 'tissue_ontology_term_id', 'development_stage_ontology_term_id', 'disease_ontology_term_id', 'assay_ontology_term_id', 'sex_ontology_term_id', 'self_reported_ethnicity_ontology_term_id', 'is_primary_data', 'suspension_type', 'tissue_type', 'donor_id', 'cell_type', 'assay', 'disease', 'sex', 'tissue', 'self_reported_ethnicity', 'development_stage', 'observation_joinid'
    var: 'gene_symbols', 'feature_is_filtered', 'feature_name', 'feature_reference', 'feature_biotype', 'feature_length', 'feature_type'
    uns: 'citation', 'control_vs_disease_colors', 'default_embeddi

### 0.2. Removing old graphs and neighbors

This is done because the data will be subset after, so these need to be recalculated. It also improves processing speed a lot.

In [ ]:
# --- PRE-CLEANUP CHECK ---
print("="*40)
print("PRE-CLEANUP STATUS:")
print(f"  obsp keys: {list(adata.obsp.keys())}")
print(f"  uns keys:  {[k for k in adata.uns.keys() if k in ['neighbors', 'leiden', 'umap']]}")
print(f"  Is View?   {adata.is_view}")
print("="*40)

# --- THE CLEANUP ---
print("Sanitizing AnnData: Removing old graphs/neighbors...")

# 1. Clear Pairwise Matrices
if hasattr(adata, 'obsp'):
    for key in list(adata.obsp.keys()):
        print(f"  Removing from obsp: {key}")
        del adata.obsp[key]

# 2. Clear Metadata
for key in ['neighbors', 'umap', 'leiden']:
    if key in adata.uns:
        print(f"  Removing from uns:  {key}")
        del adata.uns[key]

# Specifically target the old embeddings
for key in ['X_scANVI', 'X_umap']:
    if key in adata.obsm:
        print(f"Removing stale embedding: {key}")
        del adata.obsm[key]

# Specifically targets old uns that are not needed anymore
stale_keys = ['neighbors', 'umap', 'leiden', 'pca', 'log1p']
for key in stale_keys:
    if key in adata.uns:
        del adata.uns[key]

# Break View if necessary
adata = adata.copy()

print("-" * 20)
print("POST-CLEANUP STATUS:")
print(f"  obsp keys: {list(adata.obsp.keys())}")
print(f"  uns keys:  {[k for k in adata.uns.keys() if k in ['neighbors', 'leiden', 'umap']]}")
print("="*40)

PRE-CLEANUP STATUS:
  obsp keys: ['connectivities', 'distances']
  uns keys:  ['neighbors', 'umap']
  Is View?   False
Sanitizing AnnData: Removing old graphs/neighbors...
  Removing from obsp: connectivities
  Removing from obsp: distances
  Removing from uns:  neighbors
  Removing from uns:  umap
Removing stale embedding: X_scANVI
Removing stale embedding: X_umap
--------------------
POST-CLEANUP STATUS:
  obsp keys: []
  uns keys:  []


### 0.3. Subsetting of data

As we are interested in researching IBD in juvenile and adult patients, we subset adata to only include the following categories:

- donor_disease: control, organ donor, CD, UC, pediatric IBD
- exclude fetal samples (organ donor)
- organ_groups: large and small intestine

In [120]:
# Select indices first
# Each condition is wrapped in () and separated by &
keep_cells = (adata.obs["donor_disease"].isin(["control","organ_donor","CD","UC","PIBD"])) & \
             (adata.obs["donor_category"].isin(["control","disease"])) & \
             (adata.obs["organ_groups"].isin(["Small_intestine","Large_intestine"]))

adata = adata[keep_cells].copy()

### 0.4. Managing metadata data inconsitencies

There are a few issues in the metadata (obs) that need to be addressed before moving forward with the analysis:
1. There is a mismatch in age for a few donors
2. One of the samples is actually two samples and not pooled
3. There is missing cell fraction information for some donors

In [122]:
d = adata.obs[["donorID_unified","donor_disease","sex","age_unified","is_primary_data"]].drop_duplicates()


d["donorID_unified"].nunique()
len(d.index)

# age mismatches of control vs disease, see below


145

150

In [123]:
# 1. Identify IDs that appear more than once
# 2. Filter the obs table to show only those rows
duplicated_rows = d[d["donorID_unified"].duplicated(keep=False)]

# 3. Print the first few rows to inspect
print(duplicated_rows.sort_values("donorID_unified"))

                                     donorID_unified donor_disease      sex  \
index                                                                         
AAACCTGAGCCACGTC-GSM3214202                     D143            UC  unknown   
AAACCTGAGAATTCCC-GSM3214201                     D143            UC  unknown   
AAACCTGAGGCTAGCA-GSM3214205                     D144            UC  unknown   
AAACCTGAGAATGTTG-GSM3214204                     D144            UC  unknown   
AAACCTGAGCGTTCCG-GSM3214208                     D145            UC  unknown   
AAACCTGAGAGTGAGA-GSM3214207                     D145            UC  unknown   
AAACCTGCAAGCTGAG-GSM3433583                     D146          PIBD     male   
AAACCTGAGCTTCGCG-GSM3433584                     D146          PIBD     male   
N104152_N-CATCCACCACCTTGTC-N104152_N            D169            CD     male   
I104152_N-GAACGGAGTATAGTAG-I104152_N            D169            CD     male   

                                     age_unified  i

The age mismatches can be explained the following way:
- D143, D144, D145: control samples that were taken in the same operation as the disease samples
- D169: Turned 35 before second sample was taken (remark in the metadata of original study)
- D146: These are actually two different patient samples that were pooled

Therefore we decided to proceed in the following way:
- D143, D144, D145: Match age to disease samples
- D169: Is correct, so can be left like it was
- D146: Retrieve the original donor name from "donor_id" column, rename samples correctly, add age of both individuals

In [124]:
# Match age of D143, D144, D145

adata.obs.loc[adata.obs["donorID_unified"].isin(["D143", "D144", "D145"]), "age_unified"] = "47-80"

In [125]:
# Change donorID_unified of D146 to D146A and D146B depending on the donor_id
print(adata.obs["donorID_unified"].dtype)

adata.obs.loc[adata.obs["donorID_unified"]=="D146",["donor_id"]].drop_duplicates()



category


,donor_id
index,
AAACCTGCAAGCTGAG-GSM3433583,S279
AAACCTGAGCTTCGCG-GSM3433584,S280


In [126]:
# 1. Extract the metadata as a separate, independent DataFrame
new_obs = adata.obs.copy()

# 2. Perform your label updates on the standalone DataFrame
new_labels = ["D146A", "D146B"]

# Standard check for categorical type using pandas directly
if not isinstance(new_obs["donorID_unified"].dtype, pd.CategoricalDtype):
    new_obs["donorID_unified"] = new_obs["donorID_unified"].astype('category')

# Add categories and perform the swap
new_obs["donorID_unified"] = new_obs["donorID_unified"].cat.add_categories(new_labels)

mask_a = new_obs["donor_id"] == "S279"
mask_b = new_obs["donor_id"] == "S280"

new_obs.loc[mask_a, "donorID_unified"] = "D146A"
new_obs.loc[mask_b, "donorID_unified"] = "D146B"

# Clean up
new_obs["donorID_unified"] = new_obs["donorID_unified"].cat.remove_unused_categories()

adata.obs = new_obs


In [127]:
# Define the chronological order for your specific ranges
age_order = ["0-3", "4-7", "9-12", "13-17", "18-34", "35-54", "47-80", "55-74", "75+"]

# Convert to Ordered Categorical (this handles the "0-3" registration automatically)
from pandas.api.types import CategoricalDtype
age_type = CategoricalDtype(categories=age_order, ordered=True)
adata.obs["age_unified"] = adata.obs["age_unified"].astype(age_type)

# Assign "0-3" to donor D146B specifically
donor_mask = adata.obs["donorID_unified"] == "D146B"
adata.obs.loc[donor_mask, "age_unified"] = "0-3"

# 4. Create and Print Frequency Table to validate the assigned age cat
# Because we set 'ordered=True' above, the columns here will follow our age_order
check_table = pd.crosstab(adata.obs["donorID_unified"], adata.obs["age_unified"])
print("\nDonor vs Age Verification:")
print(check_table.loc[["D146A", "D146B"]])


Donor vs Age Verification:
age_unified       0-3   4-7  9-12  13-17  18-34  35-54  47-80  55-74  75+
donorID_unified                                                          
D146A               0  3830     0      0      0      0      0      0    0
D146B            4619     0     0      0      0      0      0      0    0


In [128]:
d = adata.obs[["donorID_unified","donor_disease","sex","age_unified","is_primary_data"]].drop_duplicates()

# Identify IDs that appear more than once, Filter the obs table to show only those rows
duplicated_rows = d[d["donorID_unified"].duplicated(keep=False)]
print(duplicated_rows.sort_values("donorID_unified")) # now only D169 has different ages which is fine

                                     donorID_unified donor_disease   sex  \
index                                                                      
N104152_N-CATCCACCACCTTGTC-N104152_N            D169            CD  male   
I104152_N-GAACGGAGTATAGTAG-I104152_N            D169            CD  male   

                                     age_unified  is_primary_data  
index                                                              
N104152_N-CATCCACCACCTTGTC-N104152_N       18-34            False  
I104152_N-GAACGGAGTATAGTAG-I104152_N       35-54            False  


The age mismatch is now fixed as the only mismatch that is still present in D169 which is explained in the original publication. He just turned older over the samples. Different ages in one person might lead to problems in future analysis, but this setup leaves the opportunity for differnt strategies open.

In [129]:
# Final inspection of all mismatched donors
target_donors = ["D143", "D144", "D145", "D146A", "D146B", "D169"]

patient_cols = [
    "donorID_unified", "donor_category", "donor_disease", 
    "age_unified", "sex", "self_reported_ethnicity", 
    "development_stage", "control_vs_disease","study"
]

# Filter, drop duplicate cell entries, and print
# This shows you the "Master Profile" for these donors
donor_summary = adata.obs.loc[
    adata.obs["donorID_unified"].isin(target_donors), 
    patient_cols
].drop_duplicates()

print(donor_summary.sort_values("donorID_unified"))

                                     donorID_unified donor_category  \
index                                                                 
AAACCTGAGCCACGTC-GSM3214202                     D143        control   
AAACCTGAGAATTCCC-GSM3214201                     D143        disease   
AAACCTGAGGCTAGCA-GSM3214205                     D144        control   
AAACCTGAGAATGTTG-GSM3214204                     D144        disease   
AAACCTGAGCGTTCCG-GSM3214208                     D145        control   
AAACCTGAGAGTGAGA-GSM3214207                     D145        disease   
N104152_N-CATCCACCACCTTGTC-N104152_N            D169        control   
I104152_N-GAACGGAGTATAGTAG-I104152_N            D169        disease   
AAACCTGCAAGCTGAG-GSM3433583                    D146A        disease   
AAACCTGAGCTTCGCG-GSM3433584                    D146B        disease   

                                     donor_disease age_unified      sex  \
index                                                                   

There are also some missing cell fraction informations. In how this is different from the "Unknown" category in cell sorting is unfortunately not clear. If cell fractions become important in later analysis, these missing labels might be revisited.

In [130]:
adata.obs.loc[adata.obs["cell_fraction_unified"].isna(),["donorID_unified","cell_fraction_unified"]].drop_duplicates()

,donorID_unified,cell_fraction_unified
index,,
ACGAGCCGTCCGTCAG-1_2-GI4112_DUO_EPI_GEX,D166,NaN
AACGTTGGTTGGTGGA-1_4-GI4253_DUO_EPI_GEX,D165,NaN
AAACCTGAGGAGTACC-1_14-GI4394_DUO_EPI_GEX,D160,NaN


### 0.5. Addition of complete disease column

Right now there are two seperate columns for the patient disease status (donor_disease) and the control/disease distinction of the sample (donor_category). To include both in later batch effect correction, we need to create a new column (donor_disease_category) that combines disease information with control/disease. We also have healthy donors and organ donors which will be both summarized under healthy_control as this distinction is not relevant to our analysis.

In [131]:
# generate column containing both information about the disease status of the patient and the control/disease distinction of the sample

# Create a single combined ID
adata.obs["donor_disease_category"] = (
    adata.obs["donor_disease"].astype(str) + 
    "_" + 
    adata.obs["donor_category"].astype(str)
).astype("category")


# Create a dictionary for the mapping to keep it clean
rename_dict = {
    "organ_donor_control": "healthy_control",
    "control_control": "healthy_control"
}

# Apply the replacement
adata.obs["donor_disease_category"] = (
    adata.obs["donor_disease_category"]
    .replace(rename_dict)
    .astype("category")
)

# Verify the change
print(adata.obs["donor_disease_category"].value_counts())

donor_disease_category
healthy_control    312176
CD_control         199738
CD_disease          75779
PIBD_disease        67092
UC_disease          35891
UC_control          11172
Name: count, dtype: int64


/tmp/ipykernel_883456/3073408324.py:20: FutureWarning: The behavior of Series.replace (and DataFrame.replace) with CategoricalDtype is deprecated. In a future version, replace will only be used for cases that preserve the categories. To change the categories, use ser.cat.rename_categories instead.


In [132]:
adata

AnnData object with n_obs × n_vars = 701848 × 18370
    obs: 'sampleID', 'level_1_annot', 'level_2_annot', 'level_3_annot', 'n_counts', 'cell_type_ontology_term_id', 'sourceID', 'study', 'donorID_unified', 'donor_category', 'donor_disease', 'organ_unified', 'age_unified', 'sample_type', 'sample_category', 'sample_retrieval', 'tissue_fraction', 'cell_fraction_unified', 'cell_sorting', 'organ_groups', 'control_vs_disease', 'tissue_ontology_term_id', 'development_stage_ontology_term_id', 'disease_ontology_term_id', 'assay_ontology_term_id', 'sex_ontology_term_id', 'self_reported_ethnicity_ontology_term_id', 'is_primary_data', 'suspension_type', 'tissue_type', 'donor_id', 'cell_type', 'assay', 'disease', 'sex', 'tissue', 'self_reported_ethnicity', 'development_stage', 'observation_joinid', 'donor_disease_category'
    var: 'gene_symbols', 'feature_is_filtered', 'feature_name', 'feature_reference', 'feature_biotype', 'feature_length', 'feature_type'
    uns: 'citation', 'control_vs_disease_

In [133]:
print(adata.obs["donor_disease"].value_counts())
print(adata.obs["control_vs_disease"].value_counts())

donor_disease
CD             275517
organ_donor    158169
control        154007
PIBD            67092
UC              47063
Name: count, dtype: int64
control_vs_disease
control               523086
crohns_disease         75779
pediatric_IBD          67092
ulcerative_colitis     35891
Name: count, dtype: int64


In [134]:
# 1. Generate the summary
state_summary = (
    adata.obs
    .groupby(['donor_disease', 'donor_category'], observed=True)
    .size()
    .reset_index(name='cell_count')
)

# 2. Rename cancer to CRC
state_summary['donor_disease'] = state_summary['donor_disease'].astype(str).replace('cancer', 'CRC')

# 3. Define your custom order
order_list = ["control", "organ_donor", "CD", "UC", "PIBD", "CRC"]

# 4. Apply the order and sort
state_summary['donor_disease'] = pd.Categorical(
    state_summary['donor_disease'], 
    categories=order_list, 
    ordered=True
)

# Sort by disease (order_list) and then category (control then disease)
state_summary = state_summary.sort_values(['donor_disease', 'donor_category']).reset_index(drop=True)

print(state_summary)

  donor_disease donor_category  cell_count
0       control        control      154007
1   organ_donor        control      158169
2            CD        control      199738
3            CD        disease       75779
4            UC        control       11172
5            UC        disease       35891
6          PIBD        disease       67092


### 0.6. Quality control measures

To ensure a clean batch correction, we will filter out samples that contain less than 3 cells and genes that are expressed in up to 3 cells.

In [135]:
# Filter samples with < 3 cells
counts = adata.obs['sampleID'].value_counts()
keep = counts[counts >= 3].index
removed = counts[counts < 3]

# Apply filter
adata = adata[adata.obs['sampleID'].isin(keep)].copy()

# Summary
print(f"Filtered {len(removed)} samples ({removed.sum()} cells).")
if not removed.empty:
    print(f"Dropped: {removed.index.tolist()}")
print(f"Remaining: {adata.n_obs} cells, {adata.obs['sampleID'].nunique()} samples.")

Filtered 4 samples (8 cells).
Dropped: ['N11_LP_A', 'N8_LP_A', 'N20_LP_A', 'WTDAtest7844013']
Remaining: 701840 cells, 409 samples.


In [136]:
# 1. Save raw counts (from .raw.X) to a dedicated layer
adata.layers["counts"] = adata.raw.X.copy()

# Summary check
print(f"Raw counts layer now available: {list(adata.layers.keys())}")

Raw counts layer now available: ['counts']


In [ ]:

n_genes_start = adata.n_vars
print(f"Starting with {n_genes_start} genes in active workspace.")

# We pull raw integers from .raw
adata.X = adata.raw[:, adata.var_names].X.copy()
print("Successfully swapped Raw Integers into adata.X.")

# Filter on raw counts
sc.pp.filter_genes(adata, min_counts=3)
n_genes_after_filter = adata.n_vars
genes_removed = n_genes_start - n_genes_after_filter

# We need to replace our raw slot, because the mismatch in dimensions would cause the batch corr to crash
adata.raw = adata
print("Successfully swapped raw.X and var with filtered counts")

print(f"Genes removed (min_counts=3 on RAW): {genes_removed}")
print(f"Genes remaining: {n_genes_after_filter}")

# Save raw counts also in counts layer as some tools expect this format
adata.layers["counts"] = adata.X.copy()

# Normalize X for dimensionality reduction tools
sc.pp.normalize_total(adata, target_sum=1e4)
sc.pp.log1p(adata)
adata.layers["data"] = adata.X.copy()
print("X and data are now normalized slots")

# Final check
print(f"Main Object (X): {adata.shape}")

# Check all layers
for layer_name, layer_data in adata.layers.items():
    print(f"Layer '{layer_name}': {layer_data.shape}")

# Check raw, which often causes the 'hidden' shape mismatch
if adata.raw is not None:
    print(f"Raw Object: {adata.raw.shape}")

Starting with 18370 genes in active workspace.
Successfully swapped Raw Integers into adata.X.
Genes removed (min_counts=3 on RAW): 77
Genes remaining: 18293
Main Object (X): (701840, 18293)
Layer 'counts': (701840, 18293)
Layer 'data': (701840, 18293)
Raw Object: (701840, 18293)


### 0.7. PCA, Neighbors and UMAP

For later visualisation and comparison purposes we will create specially named slots for PCA, neighbors and UMAP.

In [138]:
sc.tl.pca(adata)

In [139]:
sc.pp.neighbors(adata, use_rep="X_pca", key_added="unintegrated")

In [140]:
sc.tl.umap(adata, neighbors_key="unintegrated")
adata.obsm["X_umap_unintegrated"] = adata.obsm["X_umap"].copy()
del adata.obsm["X_umap"]

In [141]:
# Final check
adata

AnnData object with n_obs × n_vars = 701840 × 18293
    obs: 'sampleID', 'level_1_annot', 'level_2_annot', 'level_3_annot', 'n_counts', 'cell_type_ontology_term_id', 'sourceID', 'study', 'donorID_unified', 'donor_category', 'donor_disease', 'organ_unified', 'age_unified', 'sample_type', 'sample_category', 'sample_retrieval', 'tissue_fraction', 'cell_fraction_unified', 'cell_sorting', 'organ_groups', 'control_vs_disease', 'tissue_ontology_term_id', 'development_stage_ontology_term_id', 'disease_ontology_term_id', 'assay_ontology_term_id', 'sex_ontology_term_id', 'self_reported_ethnicity_ontology_term_id', 'is_primary_data', 'suspension_type', 'tissue_type', 'donor_id', 'cell_type', 'assay', 'disease', 'sex', 'tissue', 'self_reported_ethnicity', 'development_stage', 'observation_joinid', 'donor_disease_category'
    var: 'gene_symbols', 'feature_is_filtered', 'feature_name', 'feature_reference', 'feature_biotype', 'feature_length', 'feature_type', 'n_counts'
    uns: 'citation', 'control

### 0.8. Renaming gene name indices

Right now the indices contain the ensembl names, however having the gene names stored in gene_symbols will be more useful for most analysis.

In [142]:
print(f"Old index gene names: {adata.var.index[:5]}")

adata.var.index = adata.var['gene_symbols']

print(f"New index gene names: {adata.var.index[:5]}")
print(f"Are the gene names unique? {adata.var.index.is_unique}")

Old index gene names: Index(['ENSG00000177757', 'ENSG00000225880', 'ENSG00000230368',
       'ENSG00000187634', 'ENSG00000188976'],
      dtype='object')
New index gene names: Index(['FAM87B', 'LINC00115', 'FAM41C', 'SAMD11', 'NOC2L'], dtype='object', name='gene_symbols')
Are the gene names unique? True


### 0.9. Saving data

The cleaned up object can now be saved for later analysis.

In [143]:
adata.write_h5ad("/home/melrie/human_intestinal/data/adata_clean.h5ad", compression=None)

In [144]:
print("Success!")

Success!
